# MSCNet Baseline Training (Phase 0)

**AGMS-Net MTP Project** — Reproducing MSCNet on Alibaba Cluster Trace 2018

**Runtime**: GPU required (Runtime > Change runtime type > T4 GPU)

**Dataset**: Upload `alibaba.csv` (96 machines, 2242 timesteps, 1.8 MB)

In [ ]:
# 1. Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Clone repo
import os
if not os.path.exists('/content/Workload_Prediction_MTP'):
    !git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
else:
    !cd /content/Workload_Prediction_MTP && git pull
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('run.py exists:', os.path.exists('run.py'))

In [ ]:
# 3. Install dependencies
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

In [ ]:
# 4. Upload dataset
import os, pandas as pd
os.makedirs('dataset', exist_ok=True)
if not os.path.exists('dataset/alibaba.csv'):
    from google.colab import files
    print('Upload alibaba.csv (96 machines, ~1.8 MB):')
    uploaded = files.upload()
    for f in uploaded: os.rename(f, 'dataset/alibaba.csv')
df = pd.read_csv('dataset/alibaba.csv')
n_features = len(df.columns) - 1  # exclude date
print(f'Dataset: {df.shape}, Features: {n_features}')
print(f'Columns: {list(df.columns[:4])} ... {list(df.columns[-3:])}')

# Verify no low-variance machines
from sklearn.preprocessing import StandardScaler
import numpy as np
n_train = int(len(df) * 0.7)
train_stds = df.drop(columns=['date']).iloc[:n_train].std()
print(f'Min training std: {train_stds.min():.4f} (should be >> 1.0)')
if train_stds.min() < 1.0:
    print('WARNING: Low-variance machines detected! Run fix_low_variance.py first.')
else:
    print(f'All {n_features} machines have good variance. Ready to train!')

In [ ]:
# 5. Setup
import os
os.makedirs('checkpoints', exist_ok=True)

# Auto-detect feature count from data
import pandas as pd
df = pd.read_csv('dataset/alibaba.csv')
N_FEATURES = len(df.columns) - 1  # exclude 'date'
print(f'enc_in = dec_in = c_out = {N_FEATURES}')
print(f'Ready to train!')

In [ ]:
# 6. Train H=24
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_24 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 24 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 7. Train H=48
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_48 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 48 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 8. Train H=72
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_72 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 72 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 9. Train H=96
!python -u run.py \
    --task_name long_term_forecast --is_training 1 \
    --root_path ./dataset --data_path alibaba.csv \
    --model_id alibaba_96_96 --model MSCNet --data custom \
    --features M --freq t \
    --seq_len 96 --label_len 96 --pred_len 96 \
    --e_layers 3 --d_layers 1 --factor 3 \
    --enc_in 96 --dec_in 96 --c_out 96 \
    --des Exp --itr 1 --n_heads 4 --d_model 128 \
    --train_epochs 100 --patience 3

In [ ]:
# 10. Results
import numpy as np, glob, os

paper = {
    24: {'MSE': 0.0030, 'MAE': 0.0292},
    48: {'MSE': 0.0041, 'MAE': 0.0356},
    72: {'MSE': 0.0050, 'MAE': 0.0399},
    96: {'MSE': 0.0058, 'MAE': 0.0433},
}

print('=' * 85)
print('MSCNet Reproduction Results vs. Paper (Alibaba, L=96)')
print('=' * 85)
print(f'{"H":>4} | {"Our MSE":>10} | {"Paper MSE":>10} | {"Delta%":>7} | {"Our MAE":>10} | {"Paper MAE":>10} | {"Delta%":>7}')
print('-' * 85)

for d in sorted(glob.glob('results/*')):
    mp = os.path.join(d, 'metrics.npy')
    if os.path.exists(mp):
        m = np.load(mp)
        mae, mse = m[0], m[1]
        for h in [24, 48, 72, 96]:
            if f'pl{h}_' in d:
                p = paper[h]
                print(f'{h:>4} | {mse:>10.6f} | {p["MSE"]:>10.6f} | {(mse-p["MSE"])/p["MSE"]*100:>+6.1f}% | {mae:>10.6f} | {p["MAE"]:>10.6f} | {(mae-p["MAE"])/p["MAE"]*100:>+6.1f}%')
                break
print('=' * 85)

# Quick sanity check on predictions
for d in sorted(glob.glob('results/*')):
    pp = os.path.join(d, 'pred.npy')
    tp = os.path.join(d, 'true.npy')
    if os.path.exists(pp):
        pred = np.load(pp)
        true = np.load(tp)
        print(f'\n{os.path.basename(d)[:40]}...')
        print(f'  Pred range: [{pred.min():.2f}, {pred.max():.2f}], mean: {pred.mean():.2f}')
        print(f'  True range: [{true.min():.2f}, {true.max():.2f}], mean: {true.mean():.2f}')

In [ ]:
# 11. Print raw log
if os.path.exists('result_long_term_forecast.txt'):
    with open('result_long_term_forecast.txt') as f: print(f.read())

In [ ]:
# 12. Save to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
import shutil
save = '/content/drive/MyDrive/AGMS_Net_Results/Phase0'
os.makedirs(save, exist_ok=True)
for f in ['results', 'checkpoints']:
    if os.path.exists(f): shutil.copytree(f, f'{save}/{f}', dirs_exist_ok=True)
if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', save)
print(f'Saved to {save}')